In [ ]:
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

#  Load model
model = YOLO("yolov8n.pt")   # replace with your trained model later

#  Collect ALL images
image_list = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            full_path = os.path.join(dirname, filename)
            image_list.append(full_path)

if len(image_list) == 0:
    raise Exception(" No images found")

print(f" Total images found: {len(image_list)}")

#  OPTIONAL: sort (latest first)
image_list = sorted(image_list, key=os.path.getmtime, reverse=True)

#  LOOP THROUGH ALL IMAGES
for idx, image_path in enumerate(image_list):

    print(f"\n Processing Image {idx+1}: {image_path}")

    # Read image
    img = cv2.imread(image_path)

    if img is None:
        print(" Skipping invalid image")
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Predict
    results = model(image_path)

    for r in results:
        for box in r.boxes:

            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[cls_id]

            percentage = round(conf * 100, 2)

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Crop detected region
            crop = img[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # 🔹 Reflectivity Score
            gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
            reflectivity = np.mean(gray)
            reflectivity_score = round((reflectivity / 255) * 100, 2)

            # 🔹 Condition Logic
            if reflectivity_score > 65:
                condition = "Good"
            elif reflectivity_score > 40:
                condition = "Moderate"
            else:
                condition = "Bad"

            # Draw bounding box
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            text = f"{condition} | {percentage}% | R:{reflectivity_score}%"
            cv2.putText(img, text, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (255, 0, 0), 2)

            print(f"➡ {label} | {condition} | {percentage}% | Reflectivity: {reflectivity_score}%")

    #  Show image
    plt.figure(figsize=(8,6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Image {idx+1}")
    plt.show()

In [ ]:
pip install ultralytics
